# 2. Training — Mixture of Experts on UNSW-NB15

Loads preprocessed splits, trains a 3-expert MoE with class-weighted loss, cosine LR
annealing, and early stopping tracked on **macro F1** (better for imbalanced multiclass).


In [1]:
import os, random, pickle
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

base_dir   = os.path.join('..')
splits_dir = os.path.join(base_dir, 'data', 'splits')
models_dir = os.path.join(base_dir, 'models')


Device: cuda


## Step 1 — Load splits

In [2]:
train_df = pd.read_csv(os.path.join(splits_dir, 'train.csv'))
val_df   = pd.read_csv(os.path.join(splits_dir, 'val.csv'))

feature_cols = [c for c in train_df.columns if c != 'label']

X_train = train_df[feature_cols].values.astype(np.float32)
y_train = train_df['label'].values.astype(np.int64)

X_val = val_df[feature_cols].values.astype(np.float32)
y_val = val_df['label'].values.astype(np.int64)

num_features = X_train.shape[1]
num_classes  = len(np.unique(y_train))
print(f'Features: {num_features}  |  Classes: {num_classes}')


Features: 206  |  Classes: 10


## Step 2 — Dataset and DataLoader

In [3]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

BATCH_SIZE = 512   # larger batch → more stable gradients

train_loader = DataLoader(TabularDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(TabularDataset(X_val,   y_val),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Class weights (softened balanced)
class_weights = np.load(os.path.join(models_dir, 'class_weights.npy'))
cw_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)
print('Class weights:', np.round(class_weights, 3))


Class weights: [1.125 1.206 0.455 0.276 0.374 0.125 0.1   0.492 1.497 4.41 ]


## Step 3 — Mixture of Experts architecture

In [4]:
class MoEModel(nn.Module):
    def __init__(self, input_dim, num_classes, dropout_p=0.4):
        super().__init__()

        # Expert 1 — medium MLP
        self.expert1 = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )

        # Expert 2 — deep MLP
        self.expert2 = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512), nn.ReLU(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256), nn.ReLU(),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )

        # Expert 3 — wide + dropout (regularization expert)
        self.expert3 = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512), nn.ReLU(), nn.Dropout(dropout_p),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(dropout_p),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128), nn.ReLU(),
            nn.Linear(128, num_classes)
        )

        # Gating network
        self.gate = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 3)
        )
        self.softmax = nn.Softmax(dim=1)

    def forward(self, x):
        gates  = self.softmax(self.gate(x))           # (B, 3)
        out1   = self.expert1(x)
        out2   = self.expert2(x)
        out3   = self.expert3(x)
        stack  = torch.stack([out1, out2, out3], dim=1) # (B, 3, C)
        output = (stack * gates.unsqueeze(-1)).sum(dim=1)
        return output

model = MoEModel(num_features, num_classes).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')


Trainable parameters: 647,969


## Step 4 — Training configuration

In [5]:
MAX_EPOCHS          = 80
EARLY_STOP_PATIENCE = 12    # epochs without macro-F1 improvement
LR_INIT             = 3e-4
WEIGHT_DECAY        = 1e-4

criterion = nn.CrossEntropyLoss(weight=cw_tensor)
optimizer = optim.Adam(model.parameters(), lr=LR_INIT, weight_decay=WEIGHT_DECAY)

# CosineAnnealingLR decays LR smoothly from LR_INIT → LR_MIN over MAX_EPOCHS
# preventing sharp LR drops from destabilizing minority-class learning
scheduler = CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS, eta_min=1e-6)

print(f'Max epochs: {MAX_EPOCHS}  |  Early-stop patience (macro F1): {EARLY_STOP_PATIENCE}')
print(f'LR: {LR_INIT}  →  1e-6 (cosine)  |  Weight decay: {WEIGHT_DECAY}')


Max epochs: 80  |  Early-stop patience (macro F1): 12
LR: 0.0003  →  1e-6 (cosine)  |  Weight decay: 0.0001


## Step 5 — Training loop with macro-F1 early stopping

In [6]:
best_f1       = 0.0
best_state    = None
no_improve    = 0
history       = []

for epoch in range(1, MAX_EPOCHS + 1):
    # ── Train ─────────────────────────────────────────────────────────────
    model.train()
    train_losses = []
    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(Xb), yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        train_losses.append(loss.item())

    # ── Validate ──────────────────────────────────────────────────────────
    model.eval()
    val_losses, preds_all, true_all = [], [], []
    with torch.no_grad():
        for Xb, yb in val_loader:
            Xb, yb = Xb.to(device), yb.to(device)
            logits = model(Xb)
            val_losses.append(criterion(logits, yb).item())
            preds_all.extend(torch.argmax(logits, 1).cpu().numpy())
            true_all.extend(yb.cpu().numpy())

    t_loss   = np.mean(train_losses)
    v_loss   = np.mean(val_losses)
    v_acc    = accuracy_score(true_all, preds_all)
    v_f1_mac = f1_score(true_all, preds_all, average='macro',    zero_division=0)
    v_f1_wt  = f1_score(true_all, preds_all, average='weighted', zero_division=0)

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    history.append({
        'epoch': epoch, 'train_loss': t_loss, 'val_loss': v_loss,
        'val_acc': v_acc, 'val_f1_macro': v_f1_mac, 'val_f1_weighted': v_f1_wt,
        'lr': current_lr
    })

    print(f'Epoch {epoch:3d}: train={t_loss:.4f}  val={v_loss:.4f}  '
          f'acc={v_acc:.4f}  f1_mac={v_f1_mac:.4f}  f1_wt={v_f1_wt:.4f}  lr={current_lr:.2e}')

    # ── Early stopping on macro F1 ─────────────────────────────────────────
    if v_f1_mac > best_f1:
        best_f1    = v_f1_mac
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= EARLY_STOP_PATIENCE:
        print(f'\nEarly stopping at epoch {epoch}. Best macro F1: {best_f1:.4f}')
        break

# Restore best checkpoint
model.load_state_dict(best_state)
print(f'\nBest val macro F1: {best_f1:.4f}')


Epoch   1: train=0.2589  val=0.2142  acc=0.9688  f1_mac=0.4560  f1_wt=0.9725  lr=3.00e-04
Epoch   2: train=0.2013  val=0.1988  acc=0.9704  f1_mac=0.4867  f1_wt=0.9735  lr=3.00e-04
Epoch   3: train=0.1913  val=0.1886  acc=0.9713  f1_mac=0.5121  f1_wt=0.9748  lr=2.99e-04
Epoch   4: train=0.1832  val=0.1818  acc=0.9725  f1_mac=0.5243  f1_wt=0.9753  lr=2.98e-04
Epoch   5: train=0.1795  val=0.1791  acc=0.9731  f1_mac=0.5353  f1_wt=0.9754  lr=2.97e-04
Epoch   6: train=0.1762  val=0.1773  acc=0.9720  f1_mac=0.5355  f1_wt=0.9750  lr=2.96e-04
Epoch   7: train=0.1742  val=0.1845  acc=0.9710  f1_mac=0.5552  f1_wt=0.9747  lr=2.94e-04
Epoch   8: train=0.1727  val=0.1728  acc=0.9725  f1_mac=0.5418  f1_wt=0.9757  lr=2.93e-04
Epoch   9: train=0.1707  val=0.1708  acc=0.9727  f1_mac=0.5724  f1_wt=0.9761  lr=2.91e-04
Epoch  10: train=0.1703  val=0.1719  acc=0.9723  f1_mac=0.5657  f1_wt=0.9757  lr=2.89e-04
Epoch  11: train=0.1692  val=0.1676  acc=0.9725  f1_mac=0.5808  f1_wt=0.9758  lr=2.86e-04
Epoch  12:

## Step 6 — Save model, history, and validation predictions

In [8]:
torch.save(model.state_dict(), os.path.join(models_dir, 'moe_model.pt'))
with open(os.path.join(models_dir, 'history.pkl'), 'wb') as f:
    pickle.dump(history, f)
np.save(os.path.join(models_dir, 'preds.npy'), np.array(preds_all))
np.save(os.path.join(models_dir, 'true.npy'),  np.array(true_all))
print('Saved: moe_model.pt  history.pkl  preds.npy  true.npy')


Saved: moe_model.pt  history.pkl  preds.npy  true.npy
